# Встреча 1 (12.10): как робот ищет путь по карте

**Лекции:** 1. Intro to Robotics и 2. Discrete planning (graph search). Задание курса к этой теме — [Lab1: реализовать A*](https://github.com/Princeton-Introduction-to-Robotics/F2026/blob/main/Lab1.ipynb).

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nikolay-Bogatyrev/princeton-intro-robotics/blob/main/project/01-graph-search.ipynb)

**Как работать с ноутбуком (30–60 минут):**
1. Запустите все ячейки: в Colab меню «Среда выполнения → Выполнить все», в Jupyter «Run → Run All Cells».
2. Покрутите ползунки в двух тренажёрах.
3. Пройдите эксперименты и запишите, что заметили.
4. В последней ячейке запишите вопросы к встрече.

Код писать не нужно: он готов, его можно читать и менять.

In [ ]:
import heapq, math, time
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown

## Главная идея

Карта помещения — это сетка клеток. Свободная клетка — вершина графа, переход в соседнюю клетку — ребро. Прямой ход стоит 1, диагональный — √2 ≈ 1.41.

Четыре классических алгоритма поиска пути устроены **одинаково**: они держат очередь клеток «к проверке» (фронтир) и раз за разом достают из неё следующую. Отличаются они только тем, **какую клетку достать первой**:

| Алгоритм | Кого достаём первым | Гарантирует кратчайший путь? |
|---|---|---|
| BFS (поиск в ширину) | того, кто раньше попал в очередь | только если все шаги одинаковой цены |
| Dijkstra | клетку с наименьшим пройденным путём `g` | да |
| Greedy (жадный) | клетку, ближайшую к цели «на глаз», `h` | нет |
| A* | клетку с наименьшим `g + w·h` | да, если `w = 1` |

`h` — эвристика, оценка оставшегося пути без учёта препятствий. Если она никогда не завышает реальную длину пути, A* находит кратчайший путь и при этом проверяет меньше клеток, чем Dijkstra.

In [ ]:
MOVES = [(1,0,1), (-1,0,1), (0,1,1), (0,-1,1),
         (1,1,math.sqrt(2)), (1,-1,math.sqrt(2)), (-1,1,math.sqrt(2)), (-1,-1,math.sqrt(2))]

def make_map(size=25, density=0.25, seed=1):
    """Случайная карта: 1 — препятствие, 0 — свободно. Старт слева внизу, цель справа вверху."""
    rng = np.random.default_rng(seed)
    grid = (rng.random((size, size)) < density).astype(int)
    start, goal = (1, 1), (size - 2, size - 2)
    for (x, y) in (start, goal):
        grid[y-1:y+2, x-1:x+2] = 0
    return grid, start, goal

def heuristic(a, b):
    """Длина пути к цели по пустой сетке с диагоналями. Никогда не завышает."""
    dx, dy = abs(a[0]-b[0]), abs(a[1]-b[1])
    return max(dx, dy) + (math.sqrt(2) - 1) * min(dx, dy)

def neighbors(grid, node):
    x, y = node
    n = grid.shape[0]
    for dx, dy, cost in MOVES:
        nx, ny = x + dx, y + dy
        if not (0 <= nx < n and 0 <= ny < n) or grid[ny, nx]:
            continue
        if dx and dy and (grid[y, nx] or grid[ny, x]):  # не срезаем угол препятствия
            continue
        yield (nx, ny), cost

def search(grid, start, goal, algo='A*', w=1.0):
    """Один код на все алгоритмы: отличается только приоритет в очереди.
    Возвращает путь, порядок проверки клеток и снимки фронтира для пошагового показа."""
    def priority(g, node):
        if algo == 'Dijkstra': return g
        if algo == 'Greedy':   return heuristic(node, goal)
        if algo == 'A*':       return g + w * heuristic(node, goal)

    g_cost, parent = {start: 0.0}, {start: None}
    if algo == 'BFS':
        frontier = deque([start])
    else:
        frontier = [(priority(0, start), 0, start)]
    counter, closed, expanded, snapshots = 0, set(), [], []

    while frontier:
        node = frontier.popleft() if algo == 'BFS' else heapq.heappop(frontier)[2]
        if node in closed:
            continue
        closed.add(node)
        expanded.append(node)
        snapshots.append([f if algo == 'BFS' else f[2] for f in frontier])
        if node == goal:
            break
        for nb, cost in neighbors(grid, node):
            new_g = g_cost[node] + cost
            if algo == 'BFS':
                if nb not in parent:
                    parent[nb], g_cost[nb] = node, new_g
                    frontier.append(nb)
            elif nb not in closed and new_g < g_cost.get(nb, math.inf):
                g_cost[nb], parent[nb] = new_g, node
                counter += 1
                heapq.heappush(frontier, (priority(new_g, nb), counter, nb))

    path = []
    if goal in closed:
        node = goal
        while node is not None:
            path.append(node)
            node = parent[node]
        path.reverse()
    length = sum(math.dist(a, b) for a, b in zip(path, path[1:]))
    return dict(path=path, length=length, expanded=expanded, snapshots=snapshots)

In [ ]:
def draw(ax, grid, start, goal, res, step=None, title=''):
    """Тёмное — препятствия, голубое — проверенные клетки, оранжевое — фронтир, зелёное — путь."""
    n_steps = len(res['expanded'])
    step = n_steps if step is None else min(step, n_steps)
    img = np.ones(grid.shape + (3,))
    img[grid == 1] = (0.25, 0.25, 0.3)
    for (x, y) in res['expanded'][:step]:
        img[y, x] = (0.7, 0.85, 1.0)
    if 0 < step < n_steps:
        for (x, y) in res['snapshots'][step - 1]:
            img[y, x] = (1.0, 0.7, 0.3)
    ax.imshow(img, origin='lower')
    if step == n_steps and res['path']:
        xs, ys = zip(*res['path'])
        ax.plot(xs, ys, color='#1a9850', lw=2.5)
    ax.plot(*start, 'o', color='#1a9850', ms=9)
    ax.plot(*goal, '*', color='#d73027', ms=14)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=10)

## Тренажёр 1. Пошагово

Двигайте ползунок `step`, чтобы посмотреть, как алгоритм «расползается» по карте. Голубые клетки уже проверены, оранжевые ждут в очереди.

In [ ]:
@interact(algo=Dropdown(options=['BFS', 'Dijkstra', 'Greedy', 'A*'], value='A*'),
          w=FloatSlider(value=1.0, min=0.0, max=5.0, step=0.5, description='w (для A*)'),
          density=FloatSlider(value=0.25, min=0.0, max=0.45, step=0.05),
          seed=IntSlider(value=1, min=1, max=50),
          step=IntSlider(value=1000, min=1, max=1000, description='step'))
def trainer_step(algo, w, density, seed, step):
    grid, start, goal = make_map(25, density, seed)
    res = search(grid, start, goal, algo, w)
    fig, ax = plt.subplots(figsize=(6, 6))
    found = f"путь {res['length']:.1f}" if res['path'] else 'пути нет'
    draw(ax, grid, start, goal, res, step,
         f"{algo}: шаг {min(step, len(res['expanded']))} из {len(res['expanded'])}, {found}")
    plt.show()

## Тренажёр 2. Все алгоритмы на одной карте

Одна и та же карта, четыре алгоритма. Сравните, **сколько клеток** проверил каждый и **насколько длинный** путь он нашёл.

In [ ]:
@interact(density=FloatSlider(value=0.25, min=0.0, max=0.45, step=0.05),
          seed=IntSlider(value=1, min=1, max=50),
          w=FloatSlider(value=1.0, min=0.0, max=5.0, step=0.5, description='w (для A*)'))
def trainer_compare(density, seed, w):
    grid, start, goal = make_map(25, density, seed)
    fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
    print(f"{'алгоритм':10} {'проверено клеток':>17} {'длина пути':>11} {'время, мс':>10}")
    for ax, algo in zip(axes, ['BFS', 'Dijkstra', 'Greedy', 'A*']):
        t0 = time.perf_counter()
        res = search(grid, start, goal, algo, w)
        ms = (time.perf_counter() - t0) * 1000
        length = f"{res['length']:.2f}" if res['path'] else '—'
        name = f'A* w={w}' if algo == 'A*' else algo
        print(f"{name:10} {len(res['expanded']):>17} {length:>11} {ms:>10.1f}")
        draw(ax, grid, start, goal, res, title=f"{name}: {len(res['expanded'])} клеток, путь {length}")
    plt.tight_layout(); plt.show()

## Эксперименты

Запишите в одну строку, что заметили. Это заготовки вопросов к встрече.

1. **BFS против Dijkstra.** На одной карте у BFS путь бывает длиннее. Почему, если BFS «гарантирует кратчайший путь»? *Подсказка: диагональный шаг дороже прямого.*
   - Заметил:
2. **Greedy.** Найдите `seed`, на котором Greedy проверил мало клеток, но нашёл заметно более длинный путь, чем A*. Когда «жадность» оправдана?
   - Заметил:
3. **Вес эвристики `w`.** В тренажёре 2 меняйте `w` от 1 до 5. Что происходит с числом проверенных клеток и с длиной пути? Чем A* при `w = 0` отличается от Dijkstra?
   - Заметил:
4. **Тупик.** Поставьте `density = 0.45` и найдите карту, где пути нет. Сколько клеток проверяет каждый алгоритм, прежде чем сдаться?
   - Заметил:
5. **Стоимость вычислений.** Если сделать карту в 10 раз мельче (сетка 250×250), как изменится число клеток? *Можно проверить: замените `25` на `100` в `make_map(...)`.*
   - Заметил:

## Мост к дрону

Сетка — упрощение. Подумайте, что она не учитывает для Crazyflie:

- Дрон не точка: у него есть размер, и препятствия надо «раздувать» на его радиус.
- Мир трёхмерный и непрерывный: сетка 3D с мелким шагом быстро раздувается до миллионов клеток (см. эксперимент 5).
- Дрон не может мгновенно повернуть на 45°: у него есть инерция и динамика (лекции 4–8).

На это отвечает лекция 3: **RRT** ищет путь в непрерывном пространстве случайными пробами, без сетки.

## Мои вопросы к встрече 12.10

1.
2.
3.